# Title Insurance Synthetic Data
Standalone native Spark notebook for realistic, FK-consistent title-insurance demo data.

In [ ]:
import argparse
import json
import math
import re

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


CATALOG = "serverless_stable_dvmvgw_catalog"
MODEL_PATH = "/Volumes/serverless_stable_dvmvgw_catalog/_metamodel/vol_root/business/title_insurance/v1/mvm/model.json"
SEED = 20261102


def parse_args():
    if "dbutils" in globals():
        defaults = {
            "catalog": CATALOG,
            "model_path": MODEL_PATH,
            "order_count": "10000",
            "months": "24",
            "seed": str(SEED),
            "table_key": "",
        }
        for name, default in defaults.items():
            try:
                dbutils.widgets.get(name)
            except Exception:
                dbutils.widgets.text(name, default)
        return argparse.Namespace(
            catalog=dbutils.widgets.get("catalog"),
            model_path=dbutils.widgets.get("model_path"),
            order_count=int(dbutils.widgets.get("order_count")),
            months=int(dbutils.widgets.get("months")),
            seed=int(dbutils.widgets.get("seed")),
            table_key=dbutils.widgets.get("table_key"),
        )
    parser = argparse.ArgumentParser()
    parser.add_argument("--catalog", default=CATALOG)
    parser.add_argument("--model-path", default=MODEL_PATH)
    parser.add_argument("--order-count", type=int, default=10000)
    parser.add_argument("--months", type=int, default=24)
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--table-key", required=True)
    return parser.parse_args()


def table_rows(product, order_count):
    name = product["table_name"]
    data_type = product.get("data_type", "").lower()
    fixed = {
        "title_order": order_count,
        "party": order_count * 2,
        "party_role": order_count * 3,
        "title_search": order_count,
        "operations_title_examination": order_count,
        "title_finding": int(order_count * 1.8),
        "title_requirement": int(order_count * 1.4),
        "title_commitment": int(order_count * 1.1),
        "title_policy": int(order_count * 1.35),
        "closing_order": int(order_count * 0.82),
        "settlement_statement": int(order_count * 0.82),
        "escrow_ledger": order_count * 4,
        "escrow_receipt": order_count * 2,
        "disbursement": order_count * 3,
        "recording_submission": int(order_count * 0.9),
        "submission_status_history": order_count * 3,
        "rejection_event": int(order_count * 0.14),
        "recorded_instrument": int(order_count * 1.8),
        "title_claim": max(500, int(order_count * 0.035)),
        "event": max(2500, int(order_count * 0.18)),
        "reserve": max(1000, int(order_count * 0.08)),
        "loss_payment": max(700, int(order_count * 0.05)),
    }
    if name in fixed:
        return fixed[name]
    if data_type == "reference_data":
        return 250
    if data_type == "master_data":
        return max(1000, order_count // 5)
    if data_type == "association_data":
        return order_count
    return max(3000, order_count // 2)


def tokens(name):
    return set(re.findall(r"[a-z0-9]+", name.lower()))


def weighted_category(row_id, values, weights):
    bucket = F.pmod(F.xxhash64(row_id, F.lit(SEED)), F.lit(sum(weights)))
    expression = None
    ceiling = 0
    for value, weight in zip(values, weights):
        ceiling += weight
        expression = F.when(bucket < ceiling, F.lit(value)) if expression is None else expression.when(
            bucket < ceiling, F.lit(value))
    return expression.otherwise(F.lit(values[-1]))


def string_value(column, row_id, table_name):
    name = column["column_name"]
    parts = tokens(name)
    if "state" in parts and "status" not in parts:
        return weighted_category(row_id, ["TX", "FL", "CA", "AZ", "NC", "GA"], [24, 18, 16, 14, 14, 14])
    if "county" in parts and "fips" not in parts:
        return weighted_category(row_id, ["Harris", "Miami-Dade", "Maricopa", "Los Angeles", "Wake", "Fulton"], [24, 18, 16, 16, 14, 12])
    if "fips" in parts:
        return weighted_category(row_id, ["48201", "12086", "04013", "06037", "37183", "13121"], [24, 18, 16, 16, 14, 12])
    if "currency" in parts:
        return F.lit("USD")
    if "channel" in parts or "production" in parts:
        return weighted_category(row_id, ["direct_branch", "centralized_lender", "independent_agent"], [45, 30, 25])
    if "transaction" in parts and ("type" in parts or "purpose" in parts):
        return weighted_category(row_id, ["purchase", "refinance", "heloc", "construction", "commercial"], [48, 30, 10, 7, 5])
    if "policy" in parts and "type" in parts:
        return weighted_category(row_id, ["owner", "lender"], [45, 55])
    if "status" in parts:
        if "claim" in table_name:
            return weighted_category(row_id, ["open", "investigating", "reserved", "closed"], [15, 20, 20, 45])
        return weighted_category(row_id, ["open", "in_progress", "complete", "cancelled"], [12, 18, 65, 5])
    if "reason" in parts and "rejection" in parts:
        return weighted_category(row_id, ["incorrect_fee", "missing_notary", "margin_violation", "legal_description_mismatch"], [36, 28, 20, 16])
    if "risk" in parts or "defect" in parts or "finding" in parts:
        return weighted_category(row_id, ["tax_lien", "judgment_lien", "vesting_issue", "easement", "chain_break"], [30, 22, 20, 18, 10])
    if "name" in parts:
        return F.concat(F.initcap(F.regexp_replace(F.lit(table_name), "_", " ")), F.lit(" "), row_id.cast("string"))
    if "email" in parts:
        return F.concat(F.lit("demo"), row_id.cast("string"), F.lit("@example.com"))
    if "phone" in parts or "fax" in parts:
        return F.concat(F.lit("555"), F.lpad(F.pmod(row_id, 10000000).cast("string"), 7, "0"))
    if "code" in parts or "number" in parts or "reference" in parts or "ref" in parts:
        return F.concat(F.upper(F.substring(F.lit(table_name), 1, 4)), F.lit("-"), F.lpad(row_id.cast("string"), 8, "0"))
    if "description" in parts or "notes" in parts or "summary" in parts:
        return F.concat(F.lit("Synthetic "), F.regexp_replace(F.lit(name), "_", " "), F.lit(" for "), row_id.cast("string"))
    return F.concat(F.lit(name + "_"), row_id.cast("string"))


def numeric_value(column, row_id):
    name = column["column_name"]
    parts = tokens(name)
    hashed = F.pmod(F.xxhash64(row_id, F.lit(name), F.lit(SEED)), F.lit(1000000))
    if any(token in parts for token in ("amount", "premium", "balance", "reserve", "payment", "fee", "loss", "liability")):
        base = F.exp(F.lit(5.5) + (hashed.cast("double") / 1000000.0) * 4.0)
        return F.round(base, 2)
    if any(token in parts for token in ("rate", "ratio", "percentage", "pct")):
        return F.round((hashed.cast("double") / 1000000.0) * 100.0, 4)
    if "days" in parts or "count" in parts or "sequence" in parts:
        return F.pmod(hashed, F.lit(45)) + 1
    return F.pmod(hashed, F.lit(10000)) + 1


def value_expression(column, row_id, start_date, months):
    dtype = column["type"].upper()
    name = column["column_name"]
    parts = tokens(name)
    if dtype == "BOOLEAN":
        return (F.pmod(F.xxhash64(row_id, F.lit(name)), F.lit(100)) < 18)
    if dtype.startswith("DATE"):
        offset = F.pmod(F.xxhash64(row_id, F.lit(name)), F.lit(months * 30))
        return F.date_add(F.lit(start_date), offset.cast("int"))
    if dtype.startswith("TIMESTAMP"):
        offset = F.pmod(F.xxhash64(row_id, F.lit(name)), F.lit(months * 30 * 86400))
        epoch = F.unix_timestamp(F.lit(start_date + " 00:00:00"), "yyyy-MM-dd HH:mm:ss")
        return F.to_timestamp(F.from_unixtime(epoch + offset), "yyyy-MM-dd HH:mm:ss")
    if dtype.startswith(("BIGINT", "INT", "SMALLINT", "TINYINT", "DOUBLE", "FLOAT", "DECIMAL")):
        value = numeric_value(column, row_id)
        if dtype.startswith("DECIMAL"):
            match = re.search(r"DECIMAL\((\d+),(\d+)\)", dtype)
            if match:
                precision, scale = int(match.group(1)), int(match.group(2))
                limit = float(10 ** (precision - scale) - 10 ** (-scale))
                value = F.greatest(F.lit(-limit), F.least(value, F.lit(limit)))
                return value.cast(DecimalType(precision, scale))
            return value.cast(dtype)
        return value.cast(dtype)
    if "country" in parts:
        return F.lit("USA")
    return string_value(column, row_id, column["table_name"]).cast(dtype)


def build_table(spark, product, domain, counts, offsets, args):
    table_name = product["table_name"]
    row_count = counts[(domain["name"], table_name)]
    partitions = max(8, min(32, math.ceil(row_count / 10000)))
    frame = spark.range(0, row_count, 1, partitions).withColumnRenamed("id", "_row_id")
    start_date = "2024-11-01"
    primary_key = product["primary_key"]
    for column in product["attributes"]:
        name = column["column_name"]
        enriched = dict(column, table_name=table_name)
        if name == primary_key:
            expression = (F.col("_row_id") + F.lit(offsets[(domain["name"], table_name)])).cast(column["type"])
        elif column.get("foreign_key_to"):
            target_domain, target_table, _ = column["foreign_key_to"].split(".")
            parent_count = counts[(target_domain, target_table)]
            parent_offset = offsets[(target_domain, target_table)]
            expression = (F.pmod(F.col("_row_id"), F.lit(parent_count)) + F.lit(parent_offset)).cast(column["type"])
        else:
            expression = value_expression(enriched, F.col("_row_id"), start_date, args.months)
        frame = frame.withColumn(name, expression)
    frame = apply_lifecycle_rules(frame, table_name)
    ordered = [attribute["column_name"] for attribute in product["attributes"]]
    return frame.select(*ordered)


def day_offset(column, modulus, minimum=0):
    return (F.pmod(F.xxhash64(column), F.lit(modulus)) + F.lit(minimum)).cast("int")


def at_time(date_column, time_text):
    return F.to_timestamp(F.concat(date_column.cast("string"), F.lit(" " + time_text)), "yyyy-MM-dd HH:mm:ss")


def apply_lifecycle_rules(frame, table_name):
    row_id = F.col("_row_id")
    if table_name == "title_order":
        scheduled = F.date_add(F.col("order_date"), day_offset(row_id, 16, 14))
        incident_delay = F.when(F.pmod(F.col("title_order_id"), 10) < 3, day_offset(row_id, 15, 7)).otherwise(day_offset(row_id, 5))
        return (frame
            .withColumn("scheduled_closing_date", scheduled)
            .withColumn("actual_closing_date", F.date_add(scheduled, incident_delay))
            .withColumn("sla_commitment_due_date", F.date_add(F.col("order_date"), 10))
            .withColumn("cancellation_date", F.when(F.col("order_status") == "cancelled", F.date_add(F.col("order_date"), day_offset(row_id, 20, 1))).cast("date"))
            .withColumn("order_received_timestamp", at_time(F.col("order_date"), "09:00:00")))
    if table_name == "recording_submission":
        submission = F.col("submission_date")
        delayed = F.pmod(F.col("title_order_id"), 10) < 3
        recording_delay = F.when(delayed, day_offset(row_id, 18, 8)).otherwise(day_offset(row_id, 6, 1))
        return (frame
            .withColumn("acceptance_date", F.date_add(submission, day_offset(row_id, 3)))
            .withColumn("recording_date", F.date_add(submission, recording_delay))
            .withColumn("resubmission_date", F.when(F.col("rejection_count") > 0, F.date_add(submission, day_offset(row_id, 8, 2))).cast("date"))
            .withColumn("last_rejection_date", F.when(F.col("rejection_count") > 0, F.date_add(submission, 1)).cast("date"))
            .withColumn("submission_timestamp", at_time(submission, "10:00:00"))
            .withColumn("recording_timestamp", at_time(F.col("recording_date"), "15:00:00")))
    if table_name == "rejection_event":
        notice = F.date_add(F.col("original_submission_date"), day_offset(row_id, 3, 1))
        received = at_time(notice, "11:00:00")
        resolution_days = day_offset(row_id, 12, 2)
        return (frame
            .withColumn("county_name", F.when(F.pmod(F.col("title_order_id"), 10) < 3, F.lit("Harris")).otherwise(F.col("county_name")))
            .withColumn("rejection_notice_date", notice)
            .withColumn("received_timestamp", received)
            .withColumn("resubmission_deadline_date", F.date_add(notice, 10))
            .withColumn("resubmission_date", F.date_add(notice, resolution_days))
            .withColumn("resolution_timestamp", F.when(F.col("rejection_status").isin("complete", "cancelled"), F.to_timestamp(F.from_unixtime(F.unix_timestamp(received) + resolution_days * 86400))).cast("timestamp"))
            .withColumn("days_open", resolution_days.cast("bigint")))
    if table_name == "title_policy":
        effective = F.col("effective_date")
        lag = F.when(F.pmod(F.col("title_order_id"), 10) < 3, day_offset(row_id, 20, 10)).otherwise(day_offset(row_id, 7, 1))
        issue = F.date_add(effective, lag)
        return (frame
            .withColumn("issue_date", issue)
            .withColumn("policy_issuance_lag_days", lag.cast("bigint"))
            .withColumn("remittance_due_date", F.date_add(issue, 30))
            .withColumn("remittance_date", F.date_add(issue, day_offset(row_id, 35, 1)))
            .withColumn("cancellation_date", F.when(F.col("policy_status") == "cancelled", F.date_add(issue, day_offset(row_id, 90, 1))).cast("date"))
            .withColumn("effective_time", at_time(effective, "00:00:00")))
    if table_name == "title_claim":
        loss = F.col("loss_date")
        fnol = F.date_add(loss, day_offset(row_id, 10, 1))
        investigation_start = F.date_add(fnol, day_offset(row_id, 3))
        investigation_close = F.date_add(investigation_start, day_offset(row_id, 75, 10))
        closed = F.col("claim_status") == "closed"
        return (frame
            .withColumn("fnol_date", fnol)
            .withColumn("fnol_received_timestamp", at_time(fnol, "09:30:00"))
            .withColumn("investigation_start_date", investigation_start)
            .withColumn("investigation_close_date", F.when(closed, investigation_close).cast("date"))
            .withColumn("settlement_date", F.when(closed, F.date_add(investigation_close, day_offset(row_id, 20))).cast("date"))
            .withColumn("closure_date", F.when(closed, F.date_add(investigation_close, day_offset(row_id, 30, 1))).cast("date")))
    return frame


def main():
    args = parse_args()
    with open(args.model_path) as handle:
        model = json.load(handle)["model"]
    products = [(domain, product) for domain in model["domains"] for product in domain["products"]]
    counts = {(domain["name"], product["table_name"]): table_rows(product, args.order_count)
              for domain, product in products}
    offsets = {key: (index + 1) * 100000000 for index, key in enumerate(sorted(counts))}
    selected = [(domain, product) for domain, product in products
                if "%s.%s" % (domain["name"], product["table_name"]) == args.table_key]
    if len(selected) != 1:
        raise ValueError("Unknown or ambiguous table_key: %s" % args.table_key)
    domain, product = selected[0]
    frame = build_table(spark, product, domain, counts, offsets, args)
    target = "%s.%s.%s" % (args.catalog, domain["database_name"], product["table_name"])
    physical_schema = spark.table(target).schema
    frame = frame.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        for field in physical_schema.fields
    ])
    row_count = counts[(domain["name"], product["table_name"])]
    print("starting %s rows -> %s" % (row_count, target), flush=True)
    frame.write.mode("overwrite").option("overwriteSchema", "false").saveAsTable(target)
    print("generated %s rows -> %s" % (row_count, target), flush=True)


if __name__ == "__main__":
    main()
